In [ ]:
# !pip install pyodbc
# !pip install pandas datasus-dbc dbfread

## Função para ler e concatenar os .dbc em um único dataframe

In [55]:
import os
import pandas as pd
from datasus_dbc import decompress
from dbfread import DBF

def ler_dbc_seguro(caminho_arquivo):
    """Função robusta para processar arquivos .dbc com tratamento de erros"""
    try:
        # Gera caminho temporário único
        temp_dbf = os.path.join(os.getcwd(), f"temp_{os.urandom(4).hex()}.dbf")
        
        # Descompressão usando a assinatura correta
        decompress(caminho_arquivo, temp_dbf)  # Corrigido: dois argumentos
        
        # Leitura do DBF resultante
        return pd.DataFrame(DBF(temp_dbf, encoding='iso-8859-1'))
    
    except Exception as e:
        print(f"Erro crítico em {os.path.basename(caminho_arquivo)}: {str(e)}")
        return pd.DataFrame()
    
    finally:
        # Limpeza segura do arquivo temporário
        if 'temp_dbf' in locals() and os.path.exists(temp_dbf):
            os.remove(temp_dbf)

# Processamento dos arquivos
diretorio_base = os.path.join(".", "data", "HANS")
arquivos = [f for f in os.listdir(diretorio_base) if f.startswith('HANSBR') and f.endswith('.dbc')]

dados_completos = pd.concat(
    [ler_dbc_seguro(os.path.join(diretorio_base, arquivo)) for arquivo in arquivos],
    ignore_index=True
)

if not dados_completos.empty:
    print(f"Dados carregados com sucesso! ({len(dados_completos)} registros)")
    print(dados_completos.head())
else:
    print("Nenhum dado válido processado")


Dados carregados com sucesso! (984168 registros)
  TP_NOT ID_AGRAVO  DT_NOTIFIC NU_ANO SG_UF_NOT ID_MUNICIP ID_REGIONA  \
0      2      A309  2001-01-10   2001        41     410304       1359   
1      2      A309  2001-01-17   2001        41     410940       1359   
2      2      A309  2001-01-16   2001        41     410940       1359   
3      2      A309  2001-01-08   2001        41     411780       1359   
4      2      A309  2001-01-02   2001        41     411780       1359   

  ID_UNIDADE     DT_DIAG SEM_DIAG  ... AVAL_ATU_N  ESQ_ATU_N DOSE_RECEB  \
0    2741474  2000-12-10           ...          0          3        NaN   
1    2741369  2000-12-23           ...          0          3        NaN   
2    2741369  2001-01-16           ...          0          1        NaN   
3    2743116  2001-01-08           ...          0          3        NaN   
4    2743116  2000-01-11           ...          2          3        NaN   

  EPIS_RACIO DTMUDESQ CONTEXAM    DTALTA_N TPALTA_N IN_VINCUL

In [56]:
dados_completos.head()

,TP_NOT,ID_AGRAVO,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_DIAG,SEM_DIAG,...,AVAL_ATU_N,ESQ_ATU_N,DOSE_RECEB,EPIS_RACIO,DTMUDESQ,CONTEXAM,DTALTA_N,TPALTA_N,IN_VINCULA,NU_LOTE_IA
0,2,A309,2001-01-10,2001,41,410304,1359,2741474,2000-12-10,,...,0,3,NaN,,None,0.0,2002-10-02,1,0,0000000
1,2,A309,2001-01-17,2001,41,410940,1359,2741369,2000-12-23,,...,0,3,NaN,,None,2.0,2001-12-19,1,0,0000000
2,2,A309,2001-01-16,2001,41,410940,1359,2741369,2001-01-16,,...,0,1,NaN,,None,1.0,2001-06-27,1,0,0000000
3,2,A309,2001-01-08,2001,41,411780,1359,2743116,2001-01-08,,...,0,3,NaN,,None,3.0,2002-01-28,1,0,0000000
4,2,A309,2001-01-02,2001,41,411780,1359,2743116,2000-01-11,,...,2,3,NaN,,None,0.0,2002-03-14,1,0,0000000


## Salvando os dados em .csv

In [57]:
import pandas as pd
from datetime import datetime


# Gerar nome do arquivo com data atual
data_atual = datetime.now().strftime('%d_%m_%Y')
nome_arquivo = f"HANSENIASE_TOTAL_{data_atual}.csv"

path_hans = os.path.join(".\\","data", "HANS", nome_arquivo)

# Salvar o DataFrame consolidado
dados_completos.to_csv(path_hans, 
                      index=False, 
                      encoding='utf-8')

print(f"Arquivo salvo com sucesso: {nome_arquivo}")
print(f"Total de registros: {len(dados_completos):,}")
print(f"Local: {os.path.abspath(nome_arquivo)}")


Arquivo salvo com sucesso: HANSENIASE_TOTAL_28_02_2025.csv
Total de registros: 984,168
Local: d:\python\data_health\HANSENIASE_TOTAL_28_02_2025.csv


## Método para carregar o arquivo mais recente gerado

In [65]:
import os
import pandas as pd
from datetime import datetime

# Caminho da pasta
path_csv = os.path.join(".\\","data", "HANS")

# Encontrar arquivos com o padrão HANSENIASE_TOTAL_
arquivos = []
for f in os.listdir(path_csv):
    if f.startswith("HANSENIASE_TOTAL_") and f.endswith(".csv"):
        try:
            # Extrair data do nome do arquivo
            data_str = f.split('_')[-3:]  # Pega os últimos 3 elementos (dd, mm, yyyy)
            data = datetime.strptime('_'.join(data_str).replace('.csv', ''), '%d_%m_%Y')
            arquivos.append((data, f))
        except Exception as e:
            print(f"Arquivo com formato inválido: {f} - {e}")

if arquivos:
    # Ordenar arquivos pela data mais recente
    arquivos.sort(reverse=True, key=lambda x: x[0])
    
    # Pegar arquivo mais recente
    ultimo_arquivo = os.path.join(path_csv, arquivos[0][1])
    
    print(f"Carregando arquivo mais recente: {ultimo_arquivo}")
    
    # Carregar dados
    df_dados = pd.read_csv(ultimo_arquivo, encoding='utf-8', low_memory=False)
    
    print("\nColunas do DataFrame:")
    print(df_dados.columns)
    print(f"\nTotal de registros: {len(df_dados):,}")
    print(f"Data de referência: {arquivos[0][0].strftime('%d/%m/%Y')}")
else:
    print("Nenhum arquivo válido encontrado na pasta")

print("\nPrimeiras linhas do DataFrame:")
df_dados.head()



Carregando arquivo mais recente: .\data\HANS\HANSENIASE_TOTAL_28_02_2025.csv

Colunas do DataFrame:
Index(['TP_NOT', 'ID_AGRAVO', 'DT_NOTIFIC', 'NU_ANO', 'SG_UF_NOT',
       'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE', 'DT_DIAG', 'SEM_DIAG',
       'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA',
       'CS_ESCOL_N', 'SG_UF', 'ID_MN_RESI', 'ID_RG_RESI', 'ID_PAIS',
       'NDUPLIC_N', 'DT_DIGITA', 'DT_TRANSUS', 'DT_TRANSDM', 'DT_TRANSSM',
       'DT_TRANSRM', 'DT_TRANSRS', 'DT_TRANSSE', 'NU_LOTE_V', 'NU_LOTE_H',
       'CS_FLXRET', 'FLXRECEBI', 'MIGRADO_W', 'ID_OCUPA_N', 'NU_LESOES',
       'FORMACLINI', 'AVALIA_N', 'CLASSOPERA', 'MODOENTR', 'MODODETECT',
       'BACILOSCOP', 'DTINICTRAT', 'ESQ_INI_N', 'CONTREG', 'NERVOSAFET',
       'UFATUAL', 'ID_MUNI_AT', 'DT_NOTI_AT', 'ID_UNID_AT', 'UFRESAT',
       'MUNIRESAT', 'DTULTCOMP', 'CLASSATUAL', 'AVAL_ATU_N', 'ESQ_ATU_N',
       'DOSE_RECEB', 'EPIS_RACIO', 'DTMUDESQ', 'CONTEXAM', 'DTALTA_N',
       'TPALTA_N', 'IN_VINCULA', 'NU_L

,TP_NOT,ID_AGRAVO,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_DIAG,SEM_DIAG,...,AVAL_ATU_N,ESQ_ATU_N,DOSE_RECEB,EPIS_RACIO,DTMUDESQ,CONTEXAM,DTALTA_N,TPALTA_N,IN_VINCULA,NU_LOTE_IA
0,2,A309,2001-01-10,2001,41.0,410304,1359.0,2741474.0,2000-12-10,NaN,...,0,3,NaN,NaN,NaN,0.0,2002-10-02,1,0.0,0000000
1,2,A309,2001-01-17,2001,41.0,410940,1359.0,2741369.0,2000-12-23,NaN,...,0,3,NaN,NaN,NaN,2.0,2001-12-19,1,0.0,0000000
2,2,A309,2001-01-16,2001,41.0,410940,1359.0,2741369.0,2001-01-16,NaN,...,0,1,NaN,NaN,NaN,1.0,2001-06-27,1,0.0,0000000
3,2,A309,2001-01-08,2001,41.0,411780,1359.0,2743116.0,2001-01-08,NaN,...,0,3,NaN,NaN,NaN,3.0,2002-01-28,1,0.0,0000000
4,2,A309,2001-01-02,2001,41.0,411780,1359.0,2743116.0,2000-01-11,NaN,...,2,3,NaN,NaN,NaN,0.0,2002-03-14,1,0.0,0000000


In [66]:
df_dados['CS_SEXO']

0         M
1         M
2         M
3         F
4         M
         ..
984163    M
984164    F
984165    M
984166    F
984167    M
Name: CS_SEXO, Length: 984168, dtype: object

In [67]:
df_dados['CS_SEXO_CAT'] = df_dados['CS_SEXO'].map({'M': 1, 'F': 2}).fillna(0).astype(int)

In [69]:
df_dados['CS_SEXO_CAT']

0         1
1         1
2         1
3         2
4         1
         ..
984163    1
984164    2
984165    1
984166    2
984167    1
Name: CS_SEXO_CAT, Length: 984168, dtype: int32